In [11]:
# Import and initialize poolparty
import poolparty as pp 
pp.init()

# Add highlighting specifications
pp.clear_highlights()
pp.add_highlight(which='tags', style='gray')
pp.add_highlight(region='cre', which='upper', style='red')
pp.add_highlight(which='lower gap', style='bold blue')

pp.clear_pools()

template_pool = pp.from_seq('TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGG</cre>ATTACGG<bc/>AGATCGGA').named('template_pool')

mutated_pool = template_pool.mutagenize('cre',
                                        mutation_rate=0.1, 
                                        mode='hybrid', 
                                        num_hybrid_states=5, 
                                        seq_name_prefix='mut_').named('mutated_pool')

deletion_pool = template_pool.deletion_scan('cre', 
                                            deletion_length=5, 
                                            positions=slice(None, None, 5), 
                                            mode='sequential', 
                                            seq_name_prefix='del_').named('deletion_pool')

sites_pool=pp.from_seqs(['AAAAA','TTTTT'], 
                        mode='sequential', 
                        op_iter_order=-1,
                        seq_name_prefix='site_').named('sites_pool')

insertion_pool = template_pool.insertion_scan('cre', 
                                              ins_pool=sites_pool, 
                                              positions=slice(None,None,5), 
                                              replace=True, 
                                              mode='sequential', 
                                              seq_name_prefix='ins_').named('insertion_pool')

combo_pool = pp.stack([mutated_pool, deletion_pool, insertion_pool])\
    .repeat_states(2, seq_name_prefix='v', op_iter_order=-2)\
    .insert_kmers('bc', length=5)\
    .named('combo_pool')\

combo_pool.print_library()

combo_pool: seq_length=None, num_values=40
state  name             seq
    0  mut_0.v0         TCCCGACT<cre>GGAAAGCaGGCAaTGAGCACcaAGG</cre>ATTACGG<bc>tggcc</bc>AGATCGGA
    1  mut_0.v1         TCCCGACT<cre>GGAAAGCaGGCAaTGAGCACcaAGG</cre>ATTACGG<bc>aaaat</bc>AGATCGGA
    2  mut_1.v0         TCCCGACT<cre>GGAAAcCGGGCAGTGAGgACACAta</cre>ATTACGG<bc>gtggt</bc>AGATCGGA
    3  mut_1.v1         TCCCGACT<cre>GGAAAcCGGGCAGTGAGgACACAta</cre>ATTACGG<bc>ggggt</bc>AGATCGGA
    4  mut_2.v0         TCCCGACT<cre>GGAAAGaGGGCAGaGAGCACAatGG</cre>ATTACGG<bc>ctgac</bc>AGATCGGA
    5  mut_2.v1         TCCCGACT<cre>GGAAAGaGGGCAGaGAGCACAatGG</cre>ATTACGG<bc>tgatg</bc>AGATCGGA
    6  mut_3.v0         TCCCGACT<cre>GGAAcGCGGGCAGTctaCACACAGG</cre>ATTACGG<bc>taata</bc>AGATCGGA
    7  mut_3.v1         TCCCGACT<cre>GGAAcGCGGGCAGTctaCACACAGG</cre>ATTACGG<bc>gaccc</bc>AGATCGGA
    8  mut_4.v0         TCCCGACT<cre>GcAAAGCGGGCAGTGAGCACACAGG</cre>ATTACGG<bc>caaaa</bc>AGATCGGA
    9  mut_4.v1         TCCCGACT<cre>GcAAAGCGGG

Pool(id=10, name='combo_pool', op='op[10]:get_kmers', num_values=40)

In [12]:
combo_pool.print_dag()

combo_pool (pool, n=40)
└── op[10]:get_kmers [mode=random, n=1]
    └── pool[9] (pool, n=40)
        └── op[9]:repeat [mode=sequential, n=2]
            └── pool[8] (pool, n=20)
                └── op[8]:stack [mode=fixed, n=3]
                    ├── mutated_pool (pool, n=5)
                    │   └── op[1]:mutagenize [mode=hybrid, n=5]
                    │       └── template_pool (pool, n=1)
                    │           └── op[0]:from_seq [mode=fixed, n=1]
                    ├── deletion_pool (pool, n=5)
                    │   └── op[3]:deletion_scan(from_seq) [mode=fixed, n=1]
                    │       └── pool[2] (pool, n=5)
                    │           └── op[2]:deletion_scan(marker_scan) [mode=sequential, n=5]
                    │               └── template_pool (pool, n=1)
                    │                   └── op[0]:from_seq [mode=fixed, n=1]
                    └── insertion_pool (pool, n=10)
                        └── op[7]:insertion_scan(replace_marker_con

Pool(id=10, name='combo_pool', op='op[10]:get_kmers', num_values=40)

In [13]:
df = combo_pool.generate_library(report_design_cards=True)
renamed_cols = {
    'name': 'name',
    'op[9]:repeat.state': 'v.state',
    'pool[8].state': 'cre.state',
    #'op[8]:stack.state': 'branch_state',
    'op[1]:mutagenize.state': 'mut.state',
    'op[2]:deletion_scan(marker_scan).state': 'del.state',
    'insertion_pool.state': 'ins.state',
    'op[6]:insertion_scan(marker_scan).state': 'ins_position.state',
    'op[4]:from_seqs.state': 'ins_site.state',
    
}
tmp_df = df.rename(columns=renamed_cols)[renamed_cols.values()]
display(tmp_df)

,name,v.state,cre.state,mut.state,del.state,ins.state,ins_position.state,ins_site.state
0,mut_0.v0,0,0,0,None,None,None,None
1,mut_0.v1,1,0,0,None,None,None,None
2,mut_1.v0,0,1,1,None,None,None,None
3,mut_1.v1,1,1,1,None,None,None,None
4,mut_2.v0,0,2,2,None,None,None,None
5,mut_2.v1,1,2,2,None,None,None,None
6,mut_3.v0,0,3,3,None,None,None,None
7,mut_3.v1,1,3,3,None,None,None,None
8,mut_4.v0,0,4,4,None,None,None,None
9,mut_4.v1,1,4,4,None,None,None,None


In [ ]:
from statetracker import Manager, Counter, product, stack

# Initialize manager
with Manager() as mgr:
    
    # Define leaf counters
    mut_counter = Counter(5, name='mut_counter')
    del_counter = Counter(5, name='del_counter')
    ins_site_counter = Counter(2, name='ins_site_counter')
    ins_position_counter = Counter(5, name='ins_position_counter')
    v_counter = Counter(2).named('v_counter')
    
    # Build composite counters
    ins_counter = product([ins_position_counter, ins_site_counter],
        name='ins_counter')
    cre_counter = stack([mut_counter, del_counter, ins_counter],
        name='cre_counter')
    seq_counter = product([v_counter, cre_counter],
        name='seq_counter')
    
# Print DAG
seq_counter.print_dag('minimal')

# Print sequences
seq_counter.get_iteration_df()

seq_counter (n=40)
└── [Product]
    ├── v_counter (n=2)
    └── cre_counter (n=20)
        └── [Stack]
            ├── mut_counter (n=5)
            ├── del_counter (n=5)
            └── ins_counter (n=10)
                └── [Product]
                    ├── ins_position_counter (n=5)
                    └── ins_site_counter (n=2)


,v_counter,cre_counter,mut_counter,del_counter,ins_counter,ins_position_counter,ins_site_counter
seq_counter,,,,,,,
0,0,0,0,None,None,None,None
1,1,0,0,None,None,None,None
2,0,1,1,None,None,None,None
3,1,1,1,None,None,None,None
4,0,2,2,None,None,None,None
5,1,2,2,None,None,None,None
6,0,3,3,None,None,None,None
7,1,3,3,None,None,None,None
8,0,4,4,None,None,None,None


In [ ]:
renamed_cols = {
    'name': 'name',
    'op[10]:get_kmers.key.kmer': 'bc_seq',
    'op[9]:repeat.state': 'v',
    'op[1]:mutagenize.key.positions': 'mut_positions',
    'op[1]:mutagenize.key.wt_chars': 'mut_wt_chars',
    'op[1]:mutagenize.key.mut_chars': 'mut_mut_chars',
    'op[2]:deletion_scan(marker_scan).key.start': 'del_start',
    'op[2]:deletion_scan(marker_scan).key.stop': 'del_stop',
    #'op[2]:deletion_scan(marker_scan).key.length': 'del_length',
    'op[2]:deletion_scan(marker_scan).key.region_content': 'deleted_seq',
    'op[6]:insertion_scan(marker_scan).key.start': 'ins_start',
    'op[6]:insertion_scan(marker_scan).key.stop': 'ins_stop',
    #'op[6]:insertion_scan(marker_scan).key.length': 'ins_length',
    'op[6]:insertion_scan(marker_scan).key.strand': 'ins_strand',
    'op[6]:insertion_scan(marker_scan).key.region_content': 'replaced_seq',
    'op[4]:from_seqs.key.seq_name': 'site_name',
    'sites_pool.seq': 'site_seq',
}
tmp_df = df.rename(columns=renamed_cols)[renamed_cols.values()]
display(tmp_df)